# Stratified split of the synthetic dataset by capture conditions
Each image's filename encodes the conditions used to capture it. The five conditions are: camera angle (A), distance (D), lighting (L), background (B), and clutter (C). A unique combination of these five values defines a sequence — for example, A1_D1_L1_B1_C1. Sequences are kept intact across splits to prevent data leakage; images from the same sequence never appear in both training and test.

The split is designed to give a clean 60/20/20 image-level distribution while ensuring that both clutter conditions are represented across splits. This is because C1 and C2 sequences have different sequence sizes: C1 has 10 images per camera angle, C2 has 40. A random split by sequence could therefore yield a heavily skewed image-level distribution.

The assignment is done manually as follows:

  - C1 sequences (clean backgrounds) are distributed across all three splits by angle: A1 to val, A2 and A3 to test, A4 to train.
  - C2 sequences (cluttered backgrounds) are restricted to train and val, since their size would dominate the test set. Angles A1, A2, A3 go to train; the A4 C2 sequence goes to val.

The result is 960 train / 320 val / 320 test images per class — an exact 60/20/20 image-level split.

In [ ]:
import shutil
from collections import defaultdict
from pathlib import Path

input_dirs = {
    "t72": Path(r"F:\Users\Nikol\Desktop\Data\Pictures\t72syn"),
    "t80": Path(r"F:\Users\Nikol\Desktop\Data\Pictures\t80syn"),
    "t90": Path(r"F:\Users\Nikol\Desktop\Data\Pictures\t90syn"),
}
subfolders = ["A1s", "A2s", "A3s", "A4s"]
output_dir = Path(r"F:\Users\Nikol\Desktop\Data\Splits1")

SEQ_TO_SPLIT = {}

# C1 sequences: standard lighting variations, clean background.
# Distributed across all three splits by camera angle.
for angle in ["A1", "A2", "A3", "A4"]:
    for light in ["L1", "L2", "L3", "L4"]:
        key = f"{angle}_D1_{light}_B1_C1"
        if angle == "A1":
            SEQ_TO_SPLIT[key] = "val"
        elif angle in ("A2", "A3"):
            SEQ_TO_SPLIT[key] = "test"
        else:  # A4
            SEQ_TO_SPLIT[key] = "train"

# C2 sequences: cluttered background, natural lighting (LN), B2 background.
# Restricted to train and val to avoid swamping the test set.
SEQ_TO_SPLIT["A1_D1_LN_B2_C2"] = "train"
SEQ_TO_SPLIT["A2_D1_LN_B2_C2"] = "train"
SEQ_TO_SPLIT["A3_D1_LN_B2_C2"] = "train"
SEQ_TO_SPLIT["A4_D1_LN_B2_C2"] = "val"
# One additional C1 sequence outside the standard lighting variations.
SEQ_TO_SPLIT["A4_D1_LN_B2_C1"] = "train"


def parse_sequence_key(filename):
    """Extract the sequence key (first five underscore-separated parts)."""
    return "_".join(filename.split("_")[:5])


def get_images(folder):
    """Return all JPG/JPEG filenames in folder."""
    return [f.name for f in folder.iterdir() if f.suffix.lower() in (".jpg", ".jpeg")]


def main():
    summary = []
    total_train = total_val = total_test = 0

    for class_name, input_dir in input_dirs.items():
        class_train = class_val = class_test = 0

        for sub in subfolders:
            folder = input_dir / sub
            if not folder.exists():
                continue

            files = get_images(folder)
            if not files:
                continue

            # Group files by sequence key, then route each sequence to its split as defined in SEQ_TO_SPLIT. 
            # Sequences without an entry in the mapping are silently skipped.
            sequences = defaultdict(list)
            for f in files:
                sequences[parse_sequence_key(f)].append(f)

            split_files = {"train": [], "val": [], "test": []}
            for key, seq_files in sequences.items():
                split = SEQ_TO_SPLIT.get(key)
                if split is not None:
                    split_files[split].extend(sorted(seq_files))

            # Copy each file into its destination, prefixing the filename with the class and subfolder to keep names unique across the merged output structure.
            for split, files_for_split in split_files.items():
                dst_folder = output_dir / split / class_name
                dst_folder.mkdir(parents=True, exist_ok=True)
                for f in files_for_split:
                    shutil.copy2(folder / f, dst_folder / f"{class_name}_{sub}_{f}")

            class_train += len(split_files["train"])
            class_val += len(split_files["val"])
            class_test += len(split_files["test"])

        class_total = class_train + class_val + class_test
        if class_total > 0:
            pcts = (class_train / class_total, class_val / class_total, class_test / class_total)
        else:
            pcts = (0.0, 0.0, 0.0)

        summary.append(
            f"{class_name}: {class_train} train, {class_val} val, {class_test} test "
            f"({pcts[0]:.1%} / {pcts[1]:.1%} / {pcts[2]:.1%})"
        )
        total_train += class_train
        total_val += class_val
        total_test += class_test

    print("Distribution:")
    for line in summary:
        print(" -", line)

    overall = total_train + total_val + total_test
    if overall > 0:
        print(
            f"\nOverall: {total_train} train, {total_val} val, {total_test} test "
            f"({total_train/overall:.1%} / {total_val/overall:.1%} / {total_test/overall:.1%})"
        )
    else:
        print("\nOverall: no images processed.")


if __name__ == "__main__":
    main()

Distribution:
 - t72: 960 train, 320 val, 320 test (60.0% / 20.0% / 20.0%)
 - t80: 960 train, 320 val, 320 test (60.0% / 20.0% / 20.0%)
 - t90: 960 train, 320 val, 320 test (60.0% / 20.0% / 20.0%)

Overall: 2880 train, 960 val, 960 test (60.0% / 20.0% / 20.0%)
